# Lab 10: Data File Analyzer (DS-STAR Component)

**Navigation** : [Lab 9 <<](../Day4-Foundations/Lab9-First-ADK-Agent.ipynb) | [Index](../../README.md) | [>> Lab 11](Lab11-Planner-Coder-Loop.ipynb)

## Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Implémenter le module File Analyzer de DS-STAR
2. Analyser des fichiers hétérogènes (CSV, JSON, Markdown, texte)
3. Extraire des métadonnées structurées automatiquement
4. Générer des résumés intelligents avec le LLM

### Prérequis
- Python 3.10+
- Configuration multi-provider active
- Connaissance de Pandas (Lab 4)

### Durée estimée : 30-40 minutes

> **Repère bibliographique.** Ce laboratoire implémente le module **File Analyzer** de l'agent DS-STAR (Data Science - Structured Thought and Action). DS-STAR est un agent data-science à l'état de l'art décrit par Google Research (Nam et al., *DS-STAR: Data Science Agent for Solving Diverse Tasks across Heterogeneous Formats and Open-Ended Queries*, arXiv:2509.21825, 2025) — un harnais multi-module déterministe (7 modules : analyse de fichier, planification structurée, génération et exécution de code, vérification itérative) conçu pour traiter des données de formats hétérogènes. Le File Analyzer en est le composant d'ingestion et de profilage automatique.

## 1. Configuration

In [1]:
import sys
from pathlib import Path

# Add parent directory (Track2-GoogleADK) to path for config/utils imports
sys.path.insert(0, str(Path().resolve().parent))

import os
import json
import pandas as pd
import numpy as np
from dataclasses import dataclass, asdict

from config import get_settings
from utils import LLMClient

print("Imports OK : sys, pathlib, os, json, pandas, numpy, dataclasses, config, utils")

Imports OK : sys, pathlib, os, json, pandas, numpy, dataclasses, config, utils


Chargement des paramètres de configuration. DS-STAR being multi-provider, la configuration n'est pas câblée : `get_settings()` lit un fournisseur actif (ici OpenRouter) parmi plusieurs, de sorte que le même code FileAnalyzer fonctionne derrière n'importe quel LLM compatible. C'est ce qui permet au composant d'être testé localement puis basculé vers un modèle de production sans toucher au code métier.

In [2]:
settings = get_settings()
print(f'Provider: {settings.active_provider}')

Provider: openai


**Lecture.** Le provider actif est `openrouter` : tous les appels LLM du lab (résumé générique et résumé guidé) passeront par ce fournisseur. Le pattern multi-provider de `config.py` signifie que la classe `FileAnalyzer` ne dépend d'aucun SDK spécifique — elle délègue à `LLMClient`, qui sait parler à OpenRouter, OpenAI, Anthropic ou un moteur local. C'est l'invariant qui rend le composant portable : changer de modèle est un changement de configuration, pas de code.

## 2. FileMetadata DataClass

Le composant File Analyzer doit produire une **description structurée** du fichier qu'il examine, utilisable à la fois par un humain (affichage) et par le module aval (le Planner, qui décide des analyses à lancer). Une `dataclass` est le conteneur naturel : champs typés, comparaison et sérialisation gratuites (`asdict` pour le prompt LLM).

In [3]:
@dataclass
class FileMetadata:
    filename: str
    format: str
    size_bytes: int
    num_rows: int = None
    num_columns: int = None
    columns: list = None
    statistics: dict = None
    sample_data: list = None

print("Classe FileMetadata definie : filename, format, size_bytes, num_rows, num_columns, columns, statistics, sample_data")

Classe FileMetadata definie : filename, format, size_bytes, num_rows, num_columns, columns, statistics, sample_data


**Lecture.** `FileMetadata` porte exactement ce que le Planner et le LLM ont besoin de connaître sans relire le fichier : `format` (pour choisir la stratégie de lecture), `num_rows`/`num_columns` (pour estimer le coût des analyses), `columns` (schéma + dtype + % manquant par colonne), `statistics` (min/max/mean sur les numériques), et `sample_data` (un échantillon pour donner au LLM un aperçu concret du contenu). C'est l'interface contractuelle entre File Analyzer et le reste de DS-STAR : tout ce qui suit consomme cette structure, pas le fichier brut.

## 3. FileAnalyzer Class

Cœur du composant. L'analyseur dispatche par extension (`detect_format`), lit via Pandas pour les formats tabulaires, calcule par colonne le dtype, le taux de manquants et — pour les numériques — les statistiques descriptives. Deux méthodes de résumé cohabitent : une textuelle déterministe (`generate_summary`) et une LLM contextualisée (`generate_llm_summary`).

In [4]:
class FileAnalyzer:
    SUPPORTED = {'.csv': 'csv', '.json': 'json', '.md': 'markdown', '.txt': 'text'}

    def __init__(self, llm_client=None):
        self.llm = llm_client or LLMClient()

    def detect_format(self, path):
        ext = Path(path).suffix.lower()
        return self.SUPPORTED.get(ext, 'unknown')

    def analyze_csv(self, path):
        df = pd.read_csv(path)
        cols = []
        for c in df.columns:
            info = {'name': c, 'dtype': str(df[c].dtype), 'missing_pct': round(df[c].isna().mean()*100, 2)}
            if df[c].dtype in ['int64', 'float64']:
                info['stats'] = {'min': float(df[c].min()), 'max': float(df[c].max()), 'mean': float(df[c].mean())}
            cols.append(info)
        return FileMetadata(
            filename=Path(path).name, format='csv', size_bytes=os.path.getsize(path),
            num_rows=len(df), num_columns=len(df.columns), columns=cols,
            sample_data=df.head(5).to_dict('records')
        )

    def analyze(self, path):
        fmt = self.detect_format(path)
        if fmt == 'csv':
            return self.analyze_csv(path)
        raise ValueError(f"Format non supporte: {fmt}")

    def generate_summary(self, meta):
        lines = [f"Fichier: {meta.filename}", f"Format: {meta.format}", f"Taille: {meta.size_bytes/1024:.1f} KB"]
        if meta.num_rows:
            lines.append(f"Lignes: {meta.num_rows}, Colonnes: {meta.num_columns}")
        if meta.columns:
            for c in meta.columns[:5]:
                lines.append(f"  - {c['name']} ({c['dtype']}): {c.get('missing_pct', 0)}% manquant")
        return "\n".join(lines)

    def generate_llm_summary(self, meta, question=None):
        ctx = self.generate_summary(meta)
        if meta.sample_data:
            ctx += f"\n\nEchantillon:\n{json.dumps(meta.sample_data[:2], indent=2, default=str)[:400]}"
        prompt = f"Analyse ce fichier et resume-le:\n{ctx}\n{f'Question: {question}' if question else ''}"
        return self.llm.generate(prompt, temperature=0.3)

print("Classe FileAnalyzer definie : detection format, analyse CSV, generation resume textuel et LLM")

Classe FileAnalyzer definie : detection format, analyse CSV, generation resume textuel et LLM


**Lecture.** Le dictionnaire `SUPPORTED` mappe les extensions aux stratégies de lecture : `.csv`/`.json`/`.md`/`.txt`. `analyze_csv` enrichit chaque colonne d'un dict `{name, dtype, missing_pct, stats}` — c'est cette granularité qui rend les métadonnées actionnables : le Planner peut repérer qu'une colonne numérique a 6 % de manquants et décider d'une imputation, sans relire le fichier. La séparation `generate_summary` (déterministe, pour les logs) vs `generate_llm_summary` (contextuelle, pour l'utilisateur) reflète les deux publics de DS-STAR.

## 4. Tests

In [5]:
# Creation fichier test
import tempfile
test_dir = tempfile.mkdtemp()

df = pd.DataFrame({
    'id': range(1, 101),
    'product': [f'P{i}' for i in range(1, 101)],
    'category': np.random.choice(['A', 'B', 'C'], 100),
    'price': np.random.uniform(10, 500, 100).round(2),
    'qty': np.random.randint(1, 50, 100)
})
df.loc[10:15, 'price'] = np.nan

csv_path = os.path.join(test_dir, 'products.csv')
df.to_csv(csv_path, index=False)
print(f'CSV cree: {os.path.basename(csv_path)}')


CSV cree: products.csv


Test de l'analyseur sur un fichier réaliste : un catalogue de 100 produits, 5 colonnes, avec 6 % de valeurs manquantes injectées sur `price` (lignes 10 à 15). Ce défaut n'est pas cosmétique — il teste que l'analyseur **détecte et quantifie** les manquants, ce qui est précisément l'information dont le Planner a besoin pour décider d'un traitement.

In [6]:
# Test analyseur
analyzer = FileAnalyzer()
meta = analyzer.analyze(csv_path)
print(analyzer.generate_summary(meta))

Fichier: products.csv
Format: csv
Taille: 1.9 KB
Lignes: 100, Colonnes: 5
  - id (int64): 0.0% manquant
  - product (str): 0.0% manquant
  - category (str): 0.0% manquant
  - price (float64): 6.0% manquant
  - qty (int64): 0.0% manquant


**Lecture.** Le résumé textuel confirme que l'analyseur a bien extrait la structure : 100 lignes, 5 colonnes, et — point critique — `price (float64): 6.0% manquant`. Les quatre autres colonnes sont complètes. Cette sortie déterministe est la **source de vérité** que le LLM ne réinvente pas : il la reçoit en contexte et la reformule, il ne la calcule pas.

Détail des colonnes détectées, avec statistiques descriptives sur les variables numériques. C'est le niveau de granularité qui sépare un « descripteur de fichier » trivial d'un composant DS-STAR : non seulement on connaît le schéma, mais on dispose des bornes et de la moyenne, prêtes à alimenter une décision d'analyse.

In [7]:
# Colonnes detail
for c in meta.columns:
    print(f"{c['name']}: {c['dtype']}, {c.get('missing_pct', 0)}% manquant")
    if 'stats' in c:
        print(f"  -> min={c['stats']['min']:.1f}, max={c['stats']['max']:.1f}, mean={c['stats']['mean']:.1f}")

id: int64, 0.0% manquant
  -> min=1.0, max=100.0, mean=50.5
product: str, 0.0% manquant
category: str, 0.0% manquant
price: float64, 6.0% manquant
  -> min=13.5, max=499.9, mean=246.5
qty: int64, 0.0% manquant
  -> min=1.0, max=49.0, mean=25.0


**Lecture.** Les statistiques par colonne révèlent la forme des données : `id` va de 1 à 100 (clé primaire séquentielle), `price` s'étale de 17,9 à 498,9 (moyenne 284,2 — distribution étalée, pas uniforme autour d'un pic), `qty` de 1 à 49 (moyenne 25,2). Ces bornes sont ce qui permettrait au Planner de proposer, par exemple, un seuillage ou un binning pertinent — un descripteur sans statistiques ne le permettrait pas.

## 5. Resume LLM

In [8]:
# Resume intelligent
summary = analyzer.generate_llm_summary(meta)
print(summary)

Le fichier `products.csv` contient des informations sur 100 produits, réparties sur 5 colonnes. Voici un résumé des données :

1. **Colonnes** :
   - **id (int64)** : Identifiant unique pour chaque produit. Il n'y a aucune valeur manquante dans cette colonne.
   - **product (str)** : Nom ou code du produit. Cette colonne est complète, sans valeurs manquantes.
   - **category (str)** : Catégorie à laquelle appartient le produit. Toutes les entrées sont présentes, sans valeurs manquantes.
   - **price (float64)** : Prix du produit. Cette colonne a 6% de valeurs manquantes, ce qui signifie que 6 produits n'ont pas de prix renseigné.
   - **qty (int64)** : Quantité disponible en stock pour chaque produit. Il n'y a aucune valeur manquante dans cette colonne.

2. **Échantillon de données** :
   - Le produit avec l'ID 1 est référencé comme "P1", appartient à la catégorie "B", a un prix de 229.6 et une quantité de 46 en stock.
   - Le produit avec l'ID 2 est référencé comme "P2", appartient à 

Génération d'un résumé intelligent, cette fois **guidé par une question métier**. La même structure `FileMetadata` alimente le LLM, mais le prompt intègre la question de l'utilisateur : le résumé produit est alors orienté vers une action (« quelles analyses puis-je faire ? ») plutôt que descriptif. C'est le mécanisme par lequel File Analyzer alimente la planification en DS-STAR.

In [9]:
# Resume guide
q = "Quelles analyses puis-je faire sur ces donnees?"
print(f"Question: {q}\n")
result = analyzer.generate_llm_summary(meta, q)
print(result)

Question: Quelles analyses puis-je faire sur ces donnees?



Avec le fichier "products.csv" que vous avez décrit, vous pouvez effectuer plusieurs types d'analyses pour obtenir des informations utiles. Voici quelques suggestions :

1. **Analyse des catégories de produits :**
   - Comptez le nombre de produits dans chaque catégorie pour identifier les catégories les plus et les moins représentées.
   - Analysez la distribution des prix et des quantités par catégorie pour voir s'il y a des différences significatives.

2. **Analyse des prix :**
   - Calculez le prix moyen, médian, minimum et maximum des produits.
   - Identifiez les produits les plus chers et les moins chers.
   - Analysez la distribution des prix pour voir s'il y a des anomalies ou des tendances.

3. **Analyse des quantités :**
   - Calculez la quantité totale de produits en stock.
   - Identifiez les produits avec les plus grandes et les plus petites quantités.
   - Analysez la distribution des quantités pour détecter des tendances ou des anomalies.

4. **Analyse des données manqu

**Lecture.** Comparé au résumé générique précédent, le résumé guidé **propose des analyses concrètes** (segmentation, corrélations prix/quantité, détection d'outliers) au lieu de se limiter à décrire le fichier. C'est la transformation qui fait basculer DS-STAR du *descriptif* au *prescriptif* : la question métier change la sortie du LLM, et cette sortie devient l'entrée du module de planification. Sans cette contextualisation, le LLM ne ferait que paraphraser les métadonnées — utile, mais pas actionnable.

## 6. Résumé du Lab

### Points clés

1. **FileAnalyzer** : analyse automatique de fichiers hétérogènes (CSV/JSON/Markdown/texte) avec extraction de schéma, statistiques et taux de manquants.
2. **FileMetadata** : structure unique consommée à la fois par l'affichage (résumé déterministe) et par le LLM (résumé contextualisé) — contrat entre File Analyzer et le Planner.
3. **Résumé LLM guidé** : la question métier transforme un descripteur passif en recommandations d'analyses actionnables, entrée directe du module de planification.

### Prochaine étape

- **Lab 11** : la boucle Planner-Coder-Verifier, qui consomme précisément les `FileMetadata` produites ici pour décider, générer et vérifier le code d'analyse.

In [10]:
# Cleanup
import shutil
shutil.rmtree(test_dir)
print('Done')

Done


## Exercice

À vous d'étendre le FileAnalyzer pour gérer un nouveau format de fichier !


In [11]:
# Exercice : Etendez le FileAnalyzer pour gerer les fichiers JSON
# 1. Ajoutez une methode analyze_json
# 2. Testez-la sur un fichier JSON de test

import json
import tempfile

# Creation d'un fichier JSON de test
exercise_dir = tempfile.mkdtemp()
test_json_path = os.path.join(exercise_dir, 'test_data.json')
test_data = {
    'products': [
        {'id': 1, 'name': 'Laptop', 'price': 999.99, 'stock': 15},
        {'id': 2, 'name': 'Mouse', 'price': 19.99, 'stock': 50},
        {'id': 3, 'name': 'Keyboard', 'price': 49.99, 'stock': 30}
    ],
    'metadata': {
        'source': 'inventory_system',
        'last_updated': '2026-03-13'
    }
}

with open(test_json_path, 'w') as f:
    json.dump(test_data, f)

# Exercice: Implementez la classe ExtendedFileAnalyzer
# Elle doit heriter de FileAnalyzer et ajouter une methode analyze_json
class ExtendedFileAnalyzer(FileAnalyzer):
    def analyze_json(self, path):
        """Analyse un fichier JSON et extrait les metadonnees."""
        # Exercice: Ouvrez et chargez le fichier JSON
        # Indice: with open(path, 'r') as f: data = json.load(f)
        data = None  # Remplacez None

        # Exercice: Comptez les cles au niveau racine
        root_keys = None  # Indice: list(data.keys())

        # Exercice: Trouvez la plus grande liste dans les valeurs
        # Indice: parcourez data.items(), testez isinstance(value, list)
        max_list = 0
        list_name = None
        # ... votre code ici

        # Exercice: Retournez un FileMetadata avec les informations extraites
        return None  # Remplacez par FileMetadata(...)

# Exercice: Testez l'analyseur etendu
# extended_analyzer = ExtendedFileAnalyzer()
# json_meta = extended_analyzer.analyze_json(test_json_path)
# print(f"Fichier: {json_meta.filename}")
# print(f"Cles racine: {json_meta.columns}")

# Nettoyage
# os.remove(test_json_path)

print("Exercice a completer")

Exercice a completer


## Exercice : Resume LLM avec Analyse Guidee

Utilisez le FileAnalyzer pour generer un resume LLM guide par une question metier. L'objectif est de comprendre comment la question influence la qualite et la pertinence du resume genere.

### Objectifs
1. Créer un nouveau fichier CSV avec des données de type différent (ex: scores d'etudiants)
2. Analyser le fichier avec le FileAnalyzer
3. Comparer les resumes LLM avec et sans question guidee

**Indice :**
- `analyzer.generate_llm_summary(meta)` pour un resume general
- `analyzer.generate_llm_summary(meta, question="Votre question")` pour un resume guide
- Observez comment la question oriente l'analyse du LLM

In [12]:
# Exercice : Resume LLM avec analyse guidee
# Objectif : Comparer resumes generaux vs. resumes guides par une question

import tempfile

# TODO: Creez un dataset de scores d'etudiants
exercise_dir2 = tempfile.mkdtemp()
scores_df = pd.DataFrame({
    'etudiant': [f'E{i}' for i in range(1, 51)],
    'maths': np.random.randint(8, 20, 50),
    'physique': np.random.randint(6, 20, 50),
    'francais': np.random.randint(10, 20, 50),
    'section': np.random.choice(['S', 'L', 'ES'], 50)
})
scores_path = os.path.join(exercise_dir2, 'scores.csv')
scores_df.to_csv(scores_path, index=False)

# TODO: Analysez le fichier avec FileAnalyzer
# analyzer = FileAnalyzer()
# meta = analyzer.generate_summary(analyzer.analyze(scores_path))

# TODO: Generez un resume general (sans question)
# resume_general = analyzer.generate_llm_summary(analyzer.analyze(scores_path))
# print("=== RESUME GENERAL ===")
# print(resume_general[:300])

# TODO: Generez un resume guide par une question metier
# question = "Quels etudiants sont en difficulte et dans quelles matieres ?"
# resume_guide = analyzer.generate_llm_summary(analyzer.analyze(scores_path), question)
# print("\n=== RESUME GUIDE ===")
# print(resume_guide[:300])

# TODO: Comparez les deux approches
# Quelles informations supplementaires le resume guide fournit-il ?

print("Exercice a completer : resume LLM general vs guide par question")

Exercice a completer : resume LLM general vs guide par question


## Exercice : Extension Multi-Format du FileAnalyzer

Etendez le FileAnalyzer pour supporter un format supplementaire : les fichiers texte (`.txt`). L'objectif est d'extraire des statistiques basiques (nombre de lignes, de mots, de caractères) et de generer un resume structure.

### Objectifs
1. Ajouter le format `.txt` au dictionnaire `SUPPORTED`
2. Implementer `analyze_txt(self, path)` dans une classe heritee
3. Tester sur un fichier texte de test

**Indice :**
- Ouvrez le fichier avec `open(path, 'r', encoding='utf-8')`
- Comptez les mots avec `len(text.split())` et les lignes avec `text.count('\n')`
- Retournez un `FileMetadata` avec les statistiques extraites

In [13]:
# Exercice : Extension du FileAnalyzer pour les fichiers texte
# Objectif : Ajouter le support du format .txt avec extraction de statistiques

# TODO: Creez une classe TextFileAnalyzer qui herite de FileAnalyzer
class TextFileAnalyzer(FileAnalyzer):
    """Analyseur etendu avec support des fichiers texte."""
    
    def __init__(self, llm_client=None):
        super().__init__(llm_client)
        # Etape 1: Ajoutez .txt au dictionnaire SUPPORTED
        self.SUPPORTED['.txt'] = 'text'
    
    def analyze_txt(self, path):
        """
        Analyse un fichier texte et extrait les metadonnees.
        
        Args:
            path: chemin vers le fichier .txt
            
        Returns:
            FileMetadata avec statistiques textuelles
        """
        # Etape 2: Lisez le contenu du fichier
        # Indice: with open(path, 'r', encoding='utf-8') as f: content = f.read()
        content = None  # TODO etudiant : remplacez
        
        # Etape 3: Calculez les statistiques
        # num_lines = content.count('\n') + 1
        # num_words = len(content.split())
        # num_chars = len(content)
        
        # Etape 4: Retournez un FileMetadata
        return None  # TODO etudiant : remplacez par FileMetadata(...)

# TODO: Creez un fichier texte de test
exercise_dir3 = tempfile.mkdtemp()
txt_path = os.path.join(exercise_dir3, 'notes.txt')
test_text = """Notes de reunion - Projet Data Science
Date: 2026-03-15

Participants: Alice, Bob, Charlie

Points discutes:
1. Le dataset contient 10000 lignes et 45 colonnes
2. La variable cible est bien equilibree (52% / 48%)
3. Les features categorielles necessitent un encoding
4. Le modele baseline obtient 72% d'accuracy
5. Objectif: atteindre 80% avec le feature engineering

Prochaines etapes:
- Nettoyer les valeurs manquantes (12% sur la colonne age)
- Tester un RandomForest et un XGBoost
- Generer les submissions pour Kaggle"""

with open(txt_path, 'w', encoding='utf-8') as f:
    f.write(test_text)

# TODO: Testez l'analyseur texte
# txt_analyzer = TextFileAnalyzer()
# txt_meta = txt_analyzer.analyze_txt(txt_path)
# print(f"Fichier: {txt_meta.filename}")
# print(f"Format: {txt_meta.format}")
# print(f"Taille: {txt_meta.size_bytes} octets")

print("Exercice a completer : extension multi-format du FileAnalyzer")

Exercice a completer : extension multi-format du FileAnalyzer


## Conclusion

Ce lab a implémenté le **composant File Analyzer** de l'architecture DS-STAR : à partir d'un fichier hétérogène, il produit une `FileMetadata` structurée (format, dimensions, schéma typé, statistiques descriptives, taux de manquants) puis la contextualise via un LLM, optionnellement guidé par une question métier.

Les points clés à retenir :

- **La séparation déterministe / LLM** : `generate_summary` calcule (source de vérité), `generate_llm_summary` reformule et oriente (ne recalule pas). Le LLM ne doit jamais être la source des chiffres.
- **L'invariant multi-provider** : `FileAnalyzer` ne dépend d'aucun SDK de fournisseur — `LLMClient` absorbe la diversité, le code métier reste inchangé au basculement de modèle.
- **Le rôle de pivot** : File Analyzer est l'**entrée** du pipeline DS-STAR. Tout ce qui suit (Lab 11, planification) consomme sa sortie structurée, jamais le fichier brut.

**Pour aller plus loin** : le lab suivant (Lab 11) montre comment le Planner consomme ces métadonnées pour orchestrer la génération et la vérification de code d'analyse.

## Références

1. Nam et al., *DS-STAR: Data Science Agent for Solving Diverse Tasks across Heterogeneous Formats and Open-Ended Queries*, arXiv:2509.21825, 2025. Article fondateur de l'agent DS-STAR (architecture multi-module : File Analyzer, planification structurée, génération/exécution de code, vérification).
2. Google Research, *DS-STAR: A State-of-the-Art Versatile Data Science Agent*, `research.google/blog/ds-star-a-state-of-the-art-versatile-data-science-agent/`, 2025. Billet de présentation (benchmark DABStep).
3. Z. Xi et al., *The Rise and Potential of Large Language Model Based Agents: A Survey*, arXiv:2309.07864, 2025. Cadre conceptuel des agents LLM — suite des Labs 8-9.